In [2]:
import torch
import triton
import triton.language as tl

In [3]:
# residual forward: out = inp1 + inp2
# each program handles BLOCK_SIZE elements (like one CUDA block)

@triton.jit
def residual_forward_kernel(out_ptr, inp1_ptr, inp2_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N  # same as `if (idx < N)` in CUDA
    inp1 = tl.load(inp1_ptr + offsets, mask=mask)
    inp2 = tl.load(inp2_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, inp1 + inp2, mask=mask)


def residual_forward(inp1, inp2, block_size=1024):
    out = torch.empty_like(inp1)
    N = out.numel()
    grid = (triton.cdiv(N, block_size),)
    residual_forward_kernel[grid](out, inp1, inp2, N, BLOCK_SIZE=block_size)
    return out

In [4]:
# validate against torch and benchmark (same sizes as residual_forward.cu)
B, T, C = 8, 1024, 768
N = B * T * C
inp1 = torch.randn(N, device="cuda", dtype=torch.float32)
inp2 = torch.randn(N, device="cuda", dtype=torch.float32)
ref = inp1 + inp2

for block_size in [32, 64, 128, 256, 512, 1024]:
    out = residual_forward(inp1, inp2, block_size)
    torch.testing.assert_close(out, ref, atol=1e-5, rtol=1e-5)
print("All result matched. Starting benchmarks.\n")

for block_size in [32, 64, 128, 256, 512, 1024]:
    ms = triton.testing.do_bench(lambda: residual_forward(inp1, inp2, block_size))
    # 2 reads + 1 write, 4 bytes each
    bandwidth = N * 3 * 4 / ms / 1e6
    print(f"block_size {block_size:4d} | time {ms:.4f} ms | bandwidth {bandwidth:.2f} GB/s")

All result matched. Starting benchmarks.

block_size   32 | time 1.1597 ms | bandwidth 65.10 GB/s
block_size   64 | time 0.6742 ms | bandwidth 111.98 GB/s
block_size  128 | time 0.5169 ms | bandwidth 146.05 GB/s
block_size  256 | time 0.5283 ms | bandwidth 142.92 GB/s
block_size  512 | time 0.5307 ms | bandwidth 142.25 GB/s
block_size 1024 | time 0.5320 ms | bandwidth 141.92 GB/s
